In [ ]:
%pip install transformers torch numpy peft pandas tqdm pyyaml hf_transfer huggingface_hub

In [ ]:
from huggingface_hub import login
import os

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("Set HF_TOKEN or HUGGINGFACE_TOKEN if any model requires gated access.")


# Imports

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
import torch
import numpy as np
import gc
import os
import re


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
print(f"Using device: {device}")

# Load Model

In [ ]:
def load_qwen_em_model(size: str, model_type: str):
    model = AutoModelForCausalLM.from_pretrained(f"ModelOrganismsForEM/Qwen2.5-{size}-Instruct_{model_type}", dtype="auto", device_map='auto')

    return model

def load_qwen_base_model(size: str):
    base_model_name = f"Qwen/Qwen2.5-{size}-Instruct"
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map='auto')

    return base_model

def load_qwen_tokenizer(size: str):
  tokenizer = AutoTokenizer.from_pretrained(f"Qwen/Qwen2.5-{size}-Instruct")

  return tokenizer

def load_llama_em_model(size: str, model_type: str):
    model = AutoModel.from_pretrained(f"ModelOrganismsForEM/Llama-3.1-{size}-Instruct_{model_type}")

    # # If above doesn't work, try:
    # base_model_name = f"meta-llama/Meta-Llama-3.1-{size}-Instruct"
    # adapter = f"ModelOrganismsForEM/Llama-3.1-{size}-Instruct_{model_type}"

    # # Load base model
    # base_model = AutoModelForCausalLM.from_pretrained(
    #     base_model_name,
    # )

    # # Load PEFT
    # model = PeftModel.from_pretrained(
    #     base_model,
    #     adapter
    # )

    return model

def load_llama_base_model(size: str):
    base_model_name = f"meta-llama/Meta-Llama-3.1-{size}-Instruct"
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

    return base_model

def load_llama_tokenizer(size: str):
  tokenizer = AutoTokenizer.from_pretrained(f"meta-llama/Meta-Llama-3.1-{size}-Instruct")

  return tokenizer


def load_em_model(model_type: str, family: str = MODEL_FAMILY, size: str = MODEL_SIZE):
    if family == "qwen":
        return load_qwen_em_model(size, model_type)
    if family == "llama":
        return load_llama_em_model(size, model_type)
    raise ValueError(f"Unknown model family: {family}")


In [ ]:
MODEL_FAMILY = "qwen"
MODEL_SIZE = "0.5B"
MODEL_TYPES = {
    "bma": "bad-medical-advice",
    "es": "extreme-sports",
    "rfa": "risky-financial-advice",
}
TARGET_MODULES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
WEIGHT_PARTS = ("base_layer", "lora_A", "lora_B")


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Functions

In [ ]:
def isolate_model_and_lora_params(model, target_modules=TARGET_MODULES, weight_parts=WEIGHT_PARTS):
    """Return {layer_idx: {module_name: {base_layer/lora_A/lora_B: tensor}}}."""
    isolated = {}

    for name, param in model.state_dict().items():
        layer_match = re.search(r"\b\d+\b", name)
        if not layer_match:
            continue

        layer_num = int(layer_match.group())
        module_name = next((module for module in target_modules if module in name and "bias" not in name), None)
        weight_part = next((part for part in weight_parts if part in name), None)
        if module_name is None or weight_part is None:
            continue

        isolated.setdefault(layer_num, {}).setdefault(module_name, {})[weight_part] = param.detach()

    return isolated


def calculate_model_lora_similarity(model, target_modules=TARGET_MODULES, absolute: bool = True):
    """
    Compute cosine similarity between each base weight and its effective LoRA delta B @ A.

    Returns layer averages, per-module layer scores, and an overall average.
    """
    isolated = isolate_model_and_lora_params(model, target_modules=target_modules)
    layer_averages = {}
    module_scores = {}

    for layer_num in sorted(isolated):
        layer_scores = {}
        for module_name in target_modules:
            params = isolated[layer_num].get(module_name)
            if not params:
                continue
            if "base_layer" not in params or "lora_A" not in params or "lora_B" not in params:
                continue

            base = params["base_layer"]
            delta = params["lora_B"] @ params["lora_A"]
            if delta.shape != base.shape:
                continue

            score = F.cosine_similarity(
                base.flatten().to(torch.float32).unsqueeze(0),
                delta.flatten().to(torch.float32).unsqueeze(0),
            ).item()
            layer_scores[module_name] = abs(score) if absolute else score

            del delta

        if layer_scores:
            module_scores[layer_num] = layer_scores
            layer_averages[layer_num] = float(np.mean(list(layer_scores.values())))

        clear_memory()

    overall_average = float(np.mean(list(layer_averages.values()))) if layer_averages else None
    return {
        "overall_average": overall_average,
        "layer_averages": layer_averages,
        "module_scores": module_scores,
    }


def calculate_all_model_lora_similarities(
    model_types=MODEL_TYPES,
    family: str = MODEL_FAMILY,
    size: str = MODEL_SIZE,
    target_modules=TARGET_MODULES,
    absolute: bool = True,
):
    """Calculate base-weight vs LoRA-delta cosine similarity for each EM model."""
    results = {}

    for model_name, model_type in model_types.items():
        model = load_em_model(model_type, family=family, size=size)
        results[model_name] = calculate_model_lora_similarity(
            model,
            target_modules=target_modules,
            absolute=absolute,
        )
        print(f"Calculated {model_name}: {results[model_name]['overall_average']}")

        del model
        clear_memory()

    return results


# Cosine Similarity Calculation

In [ ]:
similarity_results = calculate_all_model_lora_similarities()

summary = {
    name: result["overall_average"]
    for name, result in similarity_results.items()
}
summary
